In [0]:
#%run ./env

In [0]:
#%run ./python_libraries 

In [0]:
#%run ../delta_function 

In [0]:
#%run ./load_data 

In [0]:
#%run ./transform_data 

In [0]:
if current_environment == 'preprd':
    source_catalog = "ext_mal_psql_maite_vision_board_test.public"
else:
    source_catalog = f"ext_mal_psql_maite_vision_board_{current_environment}.public"

parameters_variables = spark.table(f"{source_catalog}.parameters_variables")

In [0]:
current_process = "dim_report_parameter_mapping"

# code = cle metier stable, saisie manuellement (derivee du fichier Excel
# de reference + requete sur parameters_variables). Tous les 50 codes reels
# ont ete recuperes, y compris les 3 dernieres (Anox Kwh pci, Production
# thermique PAC Kwh et Kwh/t) obtenues via leur id_parameter_variable direct.

report_parameter_mapping_raw = [
    ("malt_weight", "Quantité de malt dégermé (t)", "Somme", 1),
    ("goods_weight", "Quantité Orge mis en trempe (t)", "Somme", 2),
    ("malt_yield_r2", "R2 humide (%)", "Moyenne", 3),
    ("process_energy_electricity_unit_consumption", "Consommation électrique production (Kwh)", "Somme", 4),
    ("process_energy_electricity_unit_lhv_consumption", "Consommation électrique production (Kwh/t)", "Moyenne", 5),
    ("steep_electricity_consumption_kwh", "Consommation électrique totale trempe  (Kwh)", "Somme", 6),
    ("steep_electricity_unit_consumption_kwh", "Consommation électrique totale trempe  (Kwh/t)", "Moyenne", 7),
    ("steep_c1_electricity_consumption_kwh", "Consommation électrique trempe cycle 1 (Kwh)", "Somme", 8),
    ("steep_c1_electricity_unit_lvh_consumption", "Consommation électrique trempe cycle 1 (Kwh/t)", "Moyenne", 9),
    ("steep_c2_electricity_consumption_kwh", "Consommation électrique trempe cycle 2 (Kwh)", "Somme", 10),
    ("steep_c2_electricity_unit_lvh_consumption", "Consommation électrique trempe cycle 2 (Kwh/t)", "Moyenne", 11),
    ("germ1_electricity_consumption_kwh", "Consommation électrique germoir 1 (Kwh)", "Somme", 12),
    ("germ1_electricity_unit_lvh_consumption", "Consommation électrique germoir 1 (Kwh/t)", "Moyenne", 13),
    ("germ2_electricity_consumption_kwh", "Consommation électrique germoir 2 (Kwh)", "Somme", 14),
    ("germ2_electricity_unit_lvh_consumption", "Consommation électrique germoir 2 (Kwh/t)", "Moyenne", 15),
    ("germ3_electricity_consumption_kwh", "Consommation électrique germoir 3 (Kwh)", "Somme", 16),
    ("germ3_electricity_unit_lvh_consumption", "Consommation électrique germoir 3 (Kwh/t)", "Moyenne", 17),
    ("germ4_electricity_consumption_kwh", "Consommation électrique germoir 4 (Kwh)", "Somme", 18),
    ("germ4_electricity_unit_lvh_consumption", "Consommation électrique germoir 4 (Kwh/t)", "Moyenne", 19),
    ("germ5_electricity_consumption_kwh", "Consommation électrique germoir 5 (Kwh)", "Somme", 20),
    ("germ5_electricity_unit_lvh_consumption", "Consommation électrique germoir 5 (Kwh/t)", "Moyenne", 21),
    ("kiln1_electricity_consumption_kwh", "Consommation électrique générale Touraille 1 (Kwh)", "Somme", 22),
    ("kiln1_electricity_unit_consumption_kwh", "Consommation électrique générale Touraille 1 (Kwh/t)", "Moyenne", 23),
    ("kiln1_inbound_fan_consumption_kwh", "Consommation électrique ventilateur Touraille 1 (Kwh)", "Somme", 24),
    ("kiln1_fan_electricity_unit_consumption_kwh", "Consommation électrique ventilateur Touraille 1 (Kwh/t)", "Moyenne", 25),
    ("kiln2_electricity_consumption_kwh", "Consommation électrique générale Touraille 2 (Kwh)", "Somme", 26),
    ("kiln2_electricity_unit_consumption_kwh", "Consommation électrique générale Touraille 2 (Kwh/t)", "Moyenne", 27),
    ("kiln2_inbound_fan_consumption_kwh", "Consommation électrique ventilateur Touraille 2 (Kwh)", "Somme", 28),
    ("kiln2_fan_electricity_unit_consumption_kwh", "Consommation électrique ventilateur Touraille 2 (Kwh/t)", "Moyenne", 29),
    ("press_inbound_electricity_energy_consumption_kwh", "Consommation Presse à granule (Kwh)", "Somme", 30),
    ("press_inbound_electricity_energy_unit_consumption_kwh", "Consommation Presse à granule (Kwh/t)", "Moyenne", 31),
    ("transfert_inbound_electricity_energy_consumption_kwh", "Consommation Transfert production (Kwh)", "Somme", 32),
    ("transfert_inbound_electricity_energy_unit_consumption_kwh", "Consommation Transfert production (Kwh/t)", "Moyenne", 33),
    ("kiln_manutention_electricity_consumption_kwh", "Consommation manutention touraille (Kwh)", "Somme", 34),
    ("kiln_manutention_electricity_unit_consumption_kwh", "Consommation manutention touraille (Kwh/t)", "Moyenne", 35),
    ("kiln1_inbound_gasburner_1_consumption_m3", "Consommation Anox 1 Touraillle 1 (m3)", "Somme", 36),
    ("kiln1_inbound_gasburner_2_consumption_m3", "Consommation Anox 2  Touraillle 1 (m3)", "Somme", 37),
    ("kiln2_inbound_gasburner_1_consumption_m3", "Consommation Anox 1  Touraillle 2 (m3)", "Somme", 38),
    ("kiln2_inbound_gasburner_2_consumption_m3", "Consommation Anox 2 Touraillle 2 (m3)", "Somme", 39),
    ("kiln_gas_consumption_m3", "Consommation des 4 anox (m3)", "Somme", 40),
    ("kiln1_inbound_gasburner_1_consumption_kwh", "Consommation Anox 1 Touraillle 1 (Kwh pci)", "Somme", 41),
    ("kiln1_inbound_gasburner_2_consumption_kwh", "Consommation Anox 2 Touraillle 1 (Kwh pci)", "Somme", 42),
    ("kiln2_inbound_gasburner_1_consumption_kwh", "Consommation Anox 1 Touraillle 2 (Kwh pci)", "Somme", 43),
    ("kiln2_inbound_gasburner_2_consumption_kwh", "Consommation Anox 2 Touraillle 2 (Kwh pci)", "Somme", 44),
    ("kiln_energy_gasburner_consumption", "Consommation des 4 anox (Kwh pci)", "Somme", 45),
    ("process_energy_gas_unit_lhv_consumption", "Consommation des 4 anox (Kwh pci/t)", "Moyenne", 46),
    ("process_inbound_heatpump_consumption_kwh", "Consommation électrique PAC (Kwh)", "Somme", 47),
    ("process_energy_heatpump_consumption", "Production thermique PAC (Kwh)", "Somme", 48),
    ("process_inbound_heatpump_performanceratio_calculated", "COP (PAC)", "Moyenne", 49),
    ("process_energy_heatpump_unit_lhv_consumption", "Production thermique PAC (Kwh/t)", "Moyenne", 50),
]

# Deux pseudo-parametres : ils portent un ATTRIBUT du batch (variete, cahier des
# charges) et non une mesure. display_order negatif pour les placer en tete des
# colonnes de la matrice, avant "Quantite de malt degerme".
#
# parameter_id sentinelle et NON null : cette colonne porte la relation 1-N vers
# fact_batch_measures cote Power BI. Le cote "1" exige des valeurs uniques et
# n'admet qu'UNE seule ligne vide -- deux null seraient vus comme un doublon et
# la relation serait refusee, cassant tout le rapport. Les valeurs negatives ne
# peuvent pas entrer en collision avec un id_parameter_variable reel et ne
# matchent aucune ligne de fact_batch_measures, ce qui est le comportement voulu.
CODES_ATTRIBUTS = ["batch_specifications", "batch_variety"]

report_parameter_attributs = [
    # (code, column_label, aggregation, display_order, parameter_id sentinelle)
    ("batch_specifications", "Spécifications", "Attribut", -2, "-2"),
    ("batch_variety",        "Variété",        "Attribut", -1, "-1"),
]

dim_report_parameter_mapping_raw = spark.createDataFrame(
    report_parameter_mapping_raw,
    ["code", "column_label", "aggregation", "display_order"]
)

In [0]:
# Jointure dynamique avec dim_parameter : recupere le parameter_id ACTUEL
# pour chaque code. Si l'ID change cote source, il est repris automatiquement
# au prochain run du pipeline, sans modification manuelle necessaire.
# La regle de gestion (aggregation) se base sur le code, qui est la cle
# metier stable -- parameter_id n'est qu'une consequence resolue dynamiquement
# pour permettre la jointure avec fact_batch_measures.
dim_report_parameter_mapping = (
    dim_report_parameter_mapping_raw.alias("a")
    .join(
        dim_parameter.alias("b"),
        F.col("a.code") == F.col("b.code"),
        "left"
    )
    .select(
        F.col("b.parameter_id"),
        F.col("a.code"),
        F.col("a.column_label"),
        F.col("a.aggregation"),
        F.col("a.display_order")
    )
)

# Ajout des pseudo-parametres d'attribut. Le parameter_id sentinelle est cast
# dans le type reel de la colonne (int ou string selon la source) pour ne pas
# casser l'union.
pid_type = dict(dim_report_parameter_mapping.dtypes)["parameter_id"]

dim_report_parameter_attributs = (
    spark.createDataFrame(
        report_parameter_attributs,
        ["code", "column_label", "aggregation", "display_order", "parameter_id_sentinelle"]
    )
    .withColumn("parameter_id", F.col("parameter_id_sentinelle").cast(pid_type))
    .select("parameter_id", "code", "column_label", "aggregation", "display_order")
)

# Union via SQL : unionByName appelle _jdf, non supporte sur cluster partage
# Unity Catalog (meme contrainte que dans transform_data et fact_batch_measures).
dim_report_parameter_mapping.createOrReplaceTempView("mapping_mesures")
dim_report_parameter_attributs.createOrReplaceTempView("mapping_attributs")

union_columns = "parameter_id, code, column_label, aggregation, display_order"

dim_report_parameter_mapping = spark.sql(f"""
    SELECT {union_columns} FROM mapping_mesures
    UNION ALL
    SELECT {union_columns} FROM mapping_attributs
""")

# Alerte si un code ne matche plus rien dans dim_parameter (ID supprime, code
# renomme cote source) : parameter_id serait null. Couvre aussi l'echec de cast
# des sentinelles. raise et non print : dans un job Databricks un print n'echoue
# pas, et la ou les colonnes concernees disparaitraient silencieusement du rapport.
nb_non_matches = dim_report_parameter_mapping.filter(F.col("parameter_id").isNull()).count()
if nb_non_matches > 0:
    raise ValueError(
        f"{nb_non_matches} ligne(s) de dim_report_parameter_mapping sans parameter_id : "
        f"la ou les colonnes correspondantes disparaitraient du rapport ENERGY V2."
    )

# Garde-fou : parameter_id porte la relation 1-N vers fact_batch_measures cote
# Power BI. Un doublon ferait echouer la relation et donc tout le rapport.
nb_doublons = (
    dim_report_parameter_mapping.groupBy("parameter_id").count()
    .filter(F.col("count") > 1).count()
)
if nb_doublons > 0:
    raise ValueError(
        f"{nb_doublons} parameter_id en doublon dans dim_report_parameter_mapping : "
        f"la relation vers fact_batch_measures serait refusee par Power BI."
    )

In [0]:
target_dim_report_parameter_mapping = current_catalog + "." + current_schema + "." + current_process
print(target_dim_report_parameter_mapping)

In [0]:
all_columns = dim_report_parameter_mapping.columns
primary_key = ['code']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
handle_table_update(
    dim_report_parameter_mapping,
    target_dim_report_parameter_mapping,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)